## Download dataset from roboflow API

In [1]:
from roboflow import Roboflow
rf = Roboflow(api_key="iKZYuqo7DE6nATajKlqs")
project = rf.workspace("wrkspc-gi0hz").project("cloud-classification-mf91q")
version = project.version(7)
dataset = version.download("tensorflow")           

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Cloud-Classification-7 in tensorflow:: 100%|█████████████████████████████████████████████████████████| 1759/1759 [00:00<00:00, 16496.58it/s]


#### Encode the class into int32

In [1]:
CLASS_NAMES = [
    "altocumulus",
    "altostratus",
    "cirrocumulus",
    "cirrostratus",
    "cirrus",
    "cumulonimbus",
    "cumulus",
    "nimbostratus",
    "stratocumulus",
    "stratus",
]

#### Prepare the dataset

In [2]:
IMG_SIZE = (224, 224)
def prepare_dataset(records, image_path):
    images = []
    targets = []
    labels = []
    for index, row in records.iterrows():
        (filename, width, height, class_name, xmin, ymin, xmax, ymax) = row
        
        fullpath = os.path.join(image_path, filename)
        img = keras.preprocessing.image.load_img(fullpath, target_size=(IMG_SIZE[0], IMG_SIZE[1]))
        img_arr = keras.preprocessing.image.img_to_array(img)
        
        # Convert into porportion of the image size
        xmin = round(xmin/ width, 2)
        ymin = round(ymin/ height, 2)
        xmax = round(xmax/ width, 2)
        ymax = round(ymax/ height, 2)
        
        images.append(img_arr)
        targets.append((xmin, ymin, xmax, ymax))
        labels.append(CLASS_NAMES.index(class_name))
    return images, targets, labels

In [3]:
import os
import pandas as pd
from tensorflow import keras

# Train set
TRAINING_CSV_FILE = 'Cloud-Classification-7/train/_annotations.csv'
TRAINING_IMAGE_DIR = 'Cloud-Classification-7/train'

training_image_records = pd.read_csv(TRAINING_CSV_FILE)

train_image_path = os.path.join(os.getcwd(), TRAINING_IMAGE_DIR)

train_images, train_targets, train_labels = prepare_dataset(training_image_records, train_image_path)

# Validate set
VALIDATING_CSV_FILE = 'Cloud-Classification-7/valid/_annotations.csv'
VALIDATING_IMAGE_DIR = 'Cloud-Classification-7/valid'

validating_image_records = pd.read_csv(VALIDATING_CSV_FILE)

valid_image_path = os.path.join(os.getcwd(), VALIDATING_IMAGE_DIR)

valid_images, valid_targets, valid_labels = prepare_dataset(validating_image_records, valid_image_path)

# Testing set
TESTING_CSV_FILE = 'Cloud-Classification-7/test/_annotations.csv'
TESTING_IMAGE_DIR = 'Cloud-Classification-7/test'

testing_image_records = pd.read_csv(TESTING_CSV_FILE)

test_image_path = os.path.join(os.getcwd(), TESTING_IMAGE_DIR)

test_images, test_targets, test_labels = prepare_dataset(testing_image_records, test_image_path)

2026-01-25 21:55:44.840018: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-25 21:55:44.847896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769352944.856900    8376 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769352944.859351    8376 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-25 21:55:44.867882: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
import numpy as np

train_images = np.array(train_images)
train_targets = np.array(train_targets)
train_labels = np.array(train_labels)

valid_images = np.array(valid_images)
valid_targets = np.array(valid_targets)
valid_labels = np.array(valid_labels)

test_images = np.array(test_images)
test_targets = np.array(test_targets)
test_labels = np.array(test_labels)

In [5]:
num_classes = len(CLASS_NAMES)

#create the common input layer
input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)

In [6]:
import tensorflow as tf

losses = {"cl_head":tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
   "bb_head":tf.keras.losses.MSE}

In [7]:
trainTargets = {
    "cl_head": train_labels,
    "bb_head": train_targets
}
validTargets = {
    "cl_head": valid_labels,
    "bb_head": valid_targets
}
training_epochs = 20

### MobileNetV2

In [8]:
base_model = keras.applications.MobileNetV2(
    include_top=False, 
    weights='imagenet', 
    input_shape=input_shape
)
base_model.trainable = True

I0000 00:00:1769352951.640799    8376 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9524 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:02:00.0, compute capability: 8.6


In [9]:
# Freezing layer
i = 0
for layer in base_model.layers[:84]:
    layer.trainable = False

In [10]:
inputs = keras.Input(shape=input_shape)
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=True)
x = keras.layers.GlobalAveragePooling2D()(x)
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

class_output = keras.layers.Dense(
    num_classes,
    activation="softmax",
    name="cl_head"
)(x)

bbox_output = keras.layers.Dense(
    4,
    activation="sigmoid",
    name="bb_head"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=[class_output, bbox_output]
)

In [11]:
model.compile(loss=losses, optimizer=keras.optimizers.Adam(learning_rate=1e-4), metrics=[['accuracy'], []])

In [12]:
from keras.callbacks import ModelCheckpoint

model_path = 'best_mobilenetv2.keras'
checkpoint = ModelCheckpoint(model_path, verbose=1, mode='max')
callbacks_list = [checkpoint]
history = model.fit(train_images, trainTargets,
             validation_data=(valid_images, validTargets),
             batch_size=32,
             epochs=training_epochs,
             shuffle=True,
             verbose=1,
             callbacks=callbacks_list
)

Epoch 1/20


/home/randi_s3nze/.conda/envs/tf/lib/python3.11/site-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
I0000 00:00:1769352958.310904    8475 service.cc:148] XLA service 0x7f4d44014920 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769352958.311001    8475 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-01-25 21:55:58.448982: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769352959.091745    8475 cuda_dnn.cc:529] Loaded cuDNN version 90101
2026-01-25 21:55:59.709047: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397]

 5/49 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - bb_head_loss: 0.2239 - cl_head_accuracy: 0.0989 - cl_head_loss: 2.6073 - loss: 2.8312

I0000 00:00:1769352967.263089    8475 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


47/49 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - bb_head_loss: 0.1490 - cl_head_accuracy: 0.2954 - cl_head_loss: 2.0653 - loss: 2.2142

2026-01-25 21:56:10.316431: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 100 bytes spill stores, 100 bytes spill loads

2026-01-25 21:56:10.389576: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 188 bytes spill stores, 188 bytes spill loads



49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - bb_head_loss: 0.1467 - cl_head_accuracy: 0.3022 - cl_head_loss: 2.0484 - loss: 2.1951

2026-01-25 21:56:19.444783: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 100 bytes spill stores, 100 bytes spill loads

2026-01-25 21:56:19.477480: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 160 bytes spill stores, 160 bytes spill loads




Epoch 1: saving model to best_mobilenetv2.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 30s 327ms/step - bb_head_loss: 0.0925 - cl_head_accuracy: 0.4615 - cl_head_loss: 1.6481 - loss: 1.7430 - val_bb_head_loss: 0.0438 - val_cl_head_accuracy: 0.4122 - val_cl_head_loss: 1.8089 - val_loss: 1.8524
Epoch 2/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - bb_head_loss: 0.0268 - cl_head_accuracy: 0.8137 - cl_head_loss: 0.7183 - loss: 0.7451
Epoch 2: saving model to best_mobilenetv2.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - bb_head_loss: 0.0248 - cl_head_accuracy: 0.8306 - cl_head_loss: 0.6599 - loss: 0.6834 - val_bb_head_loss: 0.0271 - val_cl_head_accuracy: 0.3919 - val_cl_head_loss: 1.8544 - val_loss: 1.8920
Epoch 3/20
 7/49 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - bb_head_loss: 0.0235 - cl_head_accuracy: 0.9415 - cl_head_loss: 0.3589 - loss: 0.3825

KeyboardInterrupt: 

In [ ]:
# Plot Model Loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validate'], loc='upper right')
plt.show()